
# Onset Bootstrap Sensitivity Test

This notebook tests which analysis choices drive the onset differences:

1. **Baseline correction**: subject-level vs group-level
2. **Smoothing**: Gaussian, moving average, or none
3. **Bootstrap strategy**: bootstrap group curve vs bootstrap subject onsets
4. **Threshold computation**: 2–5 SD above baseline

Outputs:
- group onset CI boxplots
- onset-difference bootstrap histograms
- one large comparison figure across all parameter combinations
- summary CSV table

The main purpose is to diagnose why one pipeline produces near-0 ms Emotion/ClipViT onsets while the other does not.


In [1]:

import os
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
from tqdm.auto import tqdm

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 300


## 1. Load RSA data

In [ ]:

# ============================================================
# Change this path if needed
# ============================================================
npz_path = r"N:\Experimental_Data\yujunchen\projects\IAPS_fMRI_EEG_fusion\Searchlight ROI EEG\rsa_eeg_fmri_essentials.npz"  # CHANGE THIS
data = np.load(npz_path)

results_dir = "results"

gistevc_rsa = np.load(
    os.path.join(results_dir, "gistevc_rsa_20260603.npy")
)

behotc_rsa = np.load(
    os.path.join(results_dir, "behotc_rsa_20260603.npy")
)

clipvit_rsa = np.load(
    os.path.join(results_dir, "clipvit_rsa_20260603.npy")
)

# ============================================================
# Data dictionary used throughout notebook
# ============================================================

rsa_dict = {
    "GIST-EVC": gistevc_rsa,
    "Emotion-OTC": behotc_rsa,
    "Scene-ClipViT": clipvit_rsa,
}

# sanity check
for name, arr in rsa_dict.items():
    print(
        f"{name}: shape={arr.shape}, "
        f"mean={np.nanmean(arr):.5f}, "
        f"std={np.nanstd(arr):.5f}"
    )

if "x_shifted" in data.files:
    x_shifted = data["x_shifted"]
else:
    x = np.arange(1, 576, 1)
    x_shifted = np.array([i * 4 - 300 for i in x])

print("Loaded:", npz_path)
for k, v in rsa_dict.items():
    print(k, v.shape)
print("x_shifted:", x_shifted.shape, x_shifted[:5], x_shifted[-5:])


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\cheny\\OneDrive - University of Florida\\BME departmental\\manuscript\\data\\eeg\\rsa_eeg_fmri_essentials.npz'

## 2. Analysis settings

In [ ]:

# ============================================================
# Windows
# ============================================================
baseline_window = (-200, 0)
test_window = (0, 1200)

# Bootstrap count.
# Use 1000 for quick testing; use 5000 or 10000 for final results.
n_boot = 1000
random_state = 42

# Conditions and comparisons
condition_names = ["GIST-EVC", "Emotion-OTC", "Scene-ClipViT"]
comparison_pairs = [
    ("GIST-EVC", "Emotion-OTC"),      # Emotion - GIST
    ("GIST-EVC", "Scene-ClipViT"),    # Scene - GIST
    ("Emotion-OTC", "Scene-ClipViT"), # Scene - Emotion
]

colors = {
    "GIST-EVC": "#5DADE2",
    "Emotion-OTC": "#58D68D",
    "Scene-ClipViT": "#EC7063",
}

# ============================================================
# Factorial parameters to test
# ============================================================
# baseline_mode:
#   subject: subtract each subject's own baseline before averaging
#   group: average subjects first, then subtract group baseline
baseline_modes = ["subject", "group"]

# smoothing_mode:
#   gaussian: gaussian_filter1d with smooth_param samples
#   moving_average: centered moving-average window with smooth_param samples
#   none: no smoothing
smoothing_settings = [
    ("gaussian", 10),
    ("moving_average", 50),
    # ("none", 0),
]

# bootstrap_strategy:
#   group_curve: resample subjects -> average curve -> detect onset
#   subject_onset: detect onset per subject -> bootstrap mean subject onset
bootstrap_strategies = ["group_curve", "subject_onset"]

# Thresholds to test
threshold_sigmas = [2, 3, 4, 5]

# Consecutive samples required for onset detection.
# Set to 1 to mimic first-threshold-crossing logic.
min_consecutive_samples = 5

param_grid = list(itertools.product(
    baseline_modes,
    smoothing_settings,
    bootstrap_strategies,
    threshold_sigmas
))

print(f"Total parameter combinations: {len(param_grid)}")


## 3. Helper functions

In [ ]:

def smooth_curve(curve, mode="gaussian", param=10):
    curve = np.asarray(curve, dtype=float)
    if mode == "none" or param is None or param == 0:
        return curve.copy()
    if mode == "gaussian":
        return gaussian_filter1d(curve, sigma=param)
    if mode == "moving_average":
        window = int(param)
        if window <= 1:
            return curve.copy()
        kernel = np.ones(window) / window
        return np.convolve(curve, kernel, mode="same")
    raise ValueError(f"Unknown smoothing mode: {mode}")


def find_first_cluster_onset(sig_mask, x_shifted, min_consecutive_samples=5):
    """Return first sample in first valid consecutive cluster."""
    start = None
    count = 0
    for i, is_sig in enumerate(sig_mask):
        if is_sig:
            if start is None:
                start = i
            count += 1
        else:
            if start is not None and count >= min_consecutive_samples:
                return x_shifted[start], start
            start = None
            count = 0
    if start is not None and count >= min_consecutive_samples:
        return x_shifted[start], start
    return np.nan, None


def prepare_curve(rsa_data, x_shifted, baseline_window=(-200, 0), baseline_mode="subject"):
    """
    Return a group curve after baseline correction.

    baseline_mode='subject': subtract each subject baseline, then average.
    baseline_mode='group': average subjects, then subtract group baseline.
    """
    baseline_mask = (x_shifted >= baseline_window[0]) & (x_shifted < baseline_window[1])

    if baseline_mode == "subject":
        subj_baseline = np.nanmean(rsa_data[:, baseline_mask], axis=1, keepdims=True)
        rsa_bc = rsa_data - subj_baseline
        group_curve = np.nanmean(rsa_bc, axis=0)
        baseline_values_for_threshold = group_curve[baseline_mask]
        return group_curve, baseline_values_for_threshold

    if baseline_mode == "group":
        group_raw = np.nanmean(rsa_data, axis=0)
        group_baseline = np.nanmean(group_raw[baseline_mask])
        group_curve = group_raw - group_baseline
        baseline_values_for_threshold = group_curve[baseline_mask]
        return group_curve, baseline_values_for_threshold

    raise ValueError(f"Unknown baseline_mode: {baseline_mode}")


def detect_group_onset(
    rsa_data,
    x_shifted,
    baseline_window=(-200, 0),
    test_window=(0, 1200),
    baseline_mode="subject",
    smoothing_mode="gaussian",
    smoothing_param=10,
    threshold_sigma=2,
    min_consecutive_samples=5,
):
    """Detect onset from a group-average curve."""
    test_mask = (x_shifted >= test_window[0]) & (x_shifted <= test_window[1])
    baseline_mask = (x_shifted >= baseline_window[0]) & (x_shifted < baseline_window[1])

    curve, _ = prepare_curve(
        rsa_data,
        x_shifted,
        baseline_window=baseline_window,
        baseline_mode=baseline_mode,
    )
    smoothed = smooth_curve(curve, mode=smoothing_mode, param=smoothing_param)

    baseline_values = smoothed[baseline_mask]
    baseline_mean = np.nanmean(baseline_values)
    baseline_std = np.nanstd(baseline_values, ddof=1)
    threshold = baseline_mean + threshold_sigma * baseline_std

    sig_mask = (smoothed > threshold) & (smoothed > 0) & test_mask
    onset_time, onset_idx = find_first_cluster_onset(
        sig_mask,
        x_shifted,
        min_consecutive_samples=min_consecutive_samples,
    )

    return {
        "onset_time": onset_time,
        "onset_idx": onset_idx,
        "curve": curve,
        "smoothed": smoothed,
        "threshold": threshold,
        "baseline_mean": baseline_mean,
        "baseline_std": baseline_std,
    }


def detect_subject_onsets(
    rsa_data,
    x_shifted,
    baseline_window=(-200, 0),
    test_window=(0, 1200),
    smoothing_mode="gaussian",
    smoothing_param=10,
    threshold_sigma=2,
    min_consecutive_samples=5,
):
    """
    Detect onsets per subject.
    Each subject is baseline-corrected and thresholded using that subject's own baseline.
    """
    baseline_mask = (x_shifted >= baseline_window[0]) & (x_shifted < baseline_window[1])
    test_mask = (x_shifted >= test_window[0]) & (x_shifted <= test_window[1])

    onsets = []
    for subj in range(rsa_data.shape[0]):
        curve = rsa_data[subj].astype(float)
        subj_baseline = np.nanmean(curve[baseline_mask])
        curve_bc = curve - subj_baseline
        smoothed = smooth_curve(curve_bc, mode=smoothing_mode, param=smoothing_param)

        baseline_values = smoothed[baseline_mask]
        baseline_mean = np.nanmean(baseline_values)
        baseline_std = np.nanstd(baseline_values, ddof=1)
        threshold = baseline_mean + threshold_sigma * baseline_std

        sig_mask = (smoothed > threshold) & (smoothed > 0) & test_mask
        onset_time, _ = find_first_cluster_onset(
            sig_mask,
            x_shifted,
            min_consecutive_samples=min_consecutive_samples,
        )
        onsets.append(onset_time)

    return np.asarray(onsets, dtype=float)


def summarize_distribution(x):
    x = np.asarray(x, dtype=float)
    valid = x[~np.isnan(x)]
    if valid.size == 0:
        return dict(ci_low=np.nan, ci_high=np.nan, mean=np.nan, median=np.nan, sd=np.nan, detection_rate=0)
    return dict(
        ci_low=np.percentile(valid, 2.5),
        ci_high=np.percentile(valid, 97.5),
        mean=np.nanmean(valid),
        median=np.nanmedian(valid),
        sd=np.nanstd(valid, ddof=1),
        detection_rate=valid.size / x.size,
    )


## 4. Bootstrap functions

In [ ]:

def bootstrap_onsets_for_condition(
    rsa_data,
    x_shifted,
    baseline_window=(-200, 0),
    test_window=(0, 1200),
    baseline_mode="subject",
    smoothing_mode="gaussian",
    smoothing_param=10,
    bootstrap_strategy="group_curve",
    threshold_sigma=2,
    min_consecutive_samples=5,
    n_boot=1000,
    random_state=42,
):
    rng = np.random.default_rng(random_state)
    n_subjects = rsa_data.shape[0]

    if bootstrap_strategy == "group_curve":
        boot_onsets = []
        for _ in range(n_boot):
            idx = rng.integers(0, n_subjects, size=n_subjects)
            boot_data = rsa_data[idx, :]
            res = detect_group_onset(
                boot_data,
                x_shifted,
                baseline_window=baseline_window,
                test_window=test_window,
                baseline_mode=baseline_mode,
                smoothing_mode=smoothing_mode,
                smoothing_param=smoothing_param,
                threshold_sigma=threshold_sigma,
                min_consecutive_samples=min_consecutive_samples,
            )
            boot_onsets.append(res["onset_time"])
        boot_onsets = np.asarray(boot_onsets, dtype=float)

        observed = detect_group_onset(
            rsa_data,
            x_shifted,
            baseline_window=baseline_window,
            test_window=test_window,
            baseline_mode=baseline_mode,
            smoothing_mode=smoothing_mode,
            smoothing_param=smoothing_param,
            threshold_sigma=threshold_sigma,
            min_consecutive_samples=min_consecutive_samples,
        )["onset_time"]
        return boot_onsets, observed

    if bootstrap_strategy == "subject_onset":
        subj_onsets = detect_subject_onsets(
            rsa_data,
            x_shifted,
            baseline_window=baseline_window,
            test_window=test_window,
            smoothing_mode=smoothing_mode,
            smoothing_param=smoothing_param,
            threshold_sigma=threshold_sigma,
            min_consecutive_samples=min_consecutive_samples,
        )
        valid_subj = subj_onsets[~np.isnan(subj_onsets)]
        if valid_subj.size == 0:
            return np.full(n_boot, np.nan), np.nan

        # Bootstrap the group-level mean onset from subject-level onset estimates.
        boot_onsets = []
        for _ in range(n_boot):
            idx = rng.integers(0, valid_subj.size, size=valid_subj.size)
            boot_onsets.append(np.nanmean(valid_subj[idx]))
        boot_onsets = np.asarray(boot_onsets, dtype=float)
        observed = np.nanmean(valid_subj)
        return boot_onsets, observed

    raise ValueError(f"Unknown bootstrap_strategy: {bootstrap_strategy}")


def run_one_configuration(
    rsa_dict,
    x_shifted,
    baseline_mode,
    smoothing_mode,
    smoothing_param,
    bootstrap_strategy,
    threshold_sigma,
    baseline_window=(-200, 0),
    test_window=(0, 1200),
    min_consecutive_samples=5,
    n_boot=1000,
    random_state=42,
):
    config_label = (
        f"baseline={baseline_mode} | smooth={smoothing_mode}:{smoothing_param} | "
        f"boot={bootstrap_strategy} | thr={threshold_sigma}SD"
    )

    onset_boot = {}
    onset_obs = {}
    onset_summary = {}

    for i, name in enumerate(condition_names):
        boot, obs = bootstrap_onsets_for_condition(
            rsa_dict[name],
            x_shifted,
            baseline_window=baseline_window,
            test_window=test_window,
            baseline_mode=baseline_mode,
            smoothing_mode=smoothing_mode,
            smoothing_param=smoothing_param,
            bootstrap_strategy=bootstrap_strategy,
            threshold_sigma=threshold_sigma,
            min_consecutive_samples=min_consecutive_samples,
            n_boot=n_boot,
            random_state=random_state + 1000 * i,
        )
        onset_boot[name] = boot
        onset_obs[name] = obs
        onset_summary[name] = summarize_distribution(boot)

    diff_boot = {}
    diff_obs = {}
    diff_summary = {}

    for a, b in comparison_pairs:
        comp_name = f"{b} - {a}"
        diffs = onset_boot[b] - onset_boot[a]
        obs_diff = onset_obs[b] - onset_obs[a]
        diff_boot[comp_name] = diffs
        diff_obs[comp_name] = obs_diff
        diff_summary[comp_name] = summarize_distribution(diffs)

    return {
        "config_label": config_label,
        "baseline_mode": baseline_mode,
        "smoothing_mode": smoothing_mode,
        "smoothing_param": smoothing_param,
        "bootstrap_strategy": bootstrap_strategy,
        "threshold_sigma": threshold_sigma,
        "onset_boot": onset_boot,
        "onset_obs": onset_obs,
        "onset_summary": onset_summary,
        "diff_boot": diff_boot,
        "diff_obs": diff_obs,
        "diff_summary": diff_summary,
    }


## 5. Run the full sensitivity analysis

In [ ]:

all_results = []

for baseline_mode, (smoothing_mode, smoothing_param), bootstrap_strategy, threshold_sigma in tqdm(param_grid):
    res = run_one_configuration(
        rsa_dict,
        x_shifted,
        baseline_mode=baseline_mode,
        smoothing_mode=smoothing_mode,
        smoothing_param=smoothing_param,
        bootstrap_strategy=bootstrap_strategy,
        threshold_sigma=threshold_sigma,
        baseline_window=baseline_window,
        test_window=test_window,
        min_consecutive_samples=min_consecutive_samples,
        n_boot=n_boot,
        random_state=random_state,
    )
    all_results.append(res)

print(f"Finished {len(all_results)} configurations")


## 6. Create summary tables

In [ ]:

rows = []

for res in all_results:
    base = {
        "baseline_mode": res["baseline_mode"],
        "smoothing_mode": res["smoothing_mode"],
        "smoothing_param": res["smoothing_param"],
        "bootstrap_strategy": res["bootstrap_strategy"],
        "threshold_sigma": res["threshold_sigma"],
        "config_label": res["config_label"],
    }

    for name in condition_names:
        s = res["onset_summary"][name]
        rows.append({
            **base,
            "measure_type": "onset",
            "measure": name,
            "observed": res["onset_obs"][name],
            **s,
            "p_one_tailed_later": np.nan,
            "p_two_tailed": np.nan,
        })

    for comp in res["diff_summary"]:
        diffs = res["diff_boot"][comp]
        valid = diffs[~np.isnan(diffs)]
        if valid.size > 0:
            p_one = np.mean(valid <= 0)   # H1: second condition later than first
            p_two = 2 * min(np.mean(valid <= 0), np.mean(valid >= 0))
            p_two = min(p_two, 1.0)
        else:
            p_one = np.nan
            p_two = np.nan
        s = res["diff_summary"][comp]
        rows.append({
            **base,
            "measure_type": "difference",
            "measure": comp,
            "observed": res["diff_obs"][comp],
            **s,
            "p_one_tailed_later": p_one,
            "p_two_tailed": p_two,
        })

summary_df = pd.DataFrame(rows)
summary_df.head(20)


In [ ]:

out_dir = "onset_sensitivity_outputs"
os.makedirs(out_dir, exist_ok=True)
summary_csv = os.path.join(out_dir, "onset_sensitivity_summary.csv")
summary_df.to_csv(summary_csv, index=False)
print("Saved:", summary_csv)


## 7. Plot one configuration in detail

In [ ]:

def plot_single_configuration(res, bins=50, save_path=None):
    fig, axes = plt.subplots(1, 4, figsize=(24, 5), constrained_layout=True)

    # ------------------------------------------------------------
    # Onset CI boxplot
    # ------------------------------------------------------------
    ax = axes[0]
    onset_dists = [res["onset_boot"][name][~np.isnan(res["onset_boot"][name])] for name in condition_names]

    box = ax.boxplot(
        onset_dists,
        labels=condition_names,
        patch_artist=True,
        widths=0.55,
        showfliers=True,
        boxprops=dict(linewidth=1.5),
        whiskerprops=dict(linewidth=1.5),
        capprops=dict(linewidth=1.5),
        medianprops=dict(color="black", linewidth=2),
        flierprops=dict(marker="o", markersize=2.5, alpha=0.4),
    )
    for patch, name in zip(box["boxes"], condition_names):
        patch.set_facecolor(colors[name])
        patch.set_alpha(0.65)

    obs = [res["onset_obs"][name] for name in condition_names]
    ax.scatter(
        [1, 2, 3],
        obs,
        color="darkred",
        s=70,
        marker="D",
        edgecolors="black",
        linewidths=1,
        zorder=10,
        label="Observed",
    )

    ax.set_title("Bootstrap onset CI", fontweight="bold")
    ax.set_ylabel("Onset time (ms)")
    ax.grid(axis="y", alpha=0.25)
    ax.tick_params(axis="x", rotation=20)
    ax.legend(fontsize=8, loc="upper left")

    # ------------------------------------------------------------
    # Difference histograms
    # ------------------------------------------------------------
    for ax, comp in zip(axes[1:], res["diff_boot"]):
        diffs = res["diff_boot"][comp]
        valid = diffs[~np.isnan(diffs)]
        summ = res["diff_summary"][comp]
        obs_diff = res["diff_obs"][comp]

        ax.hist(valid, bins=bins, color="steelblue", alpha=0.75, edgecolor="black", linewidth=0.4)
        ax.axvline(0, color="black", linestyle="--", linewidth=1.5, alpha=0.7, label="Null")
        ax.axvline(obs_diff, color="darkred", linewidth=2, label=f"Obs: {obs_diff:.0f} ms")
        ax.axvline(summ["ci_low"], color="green", linestyle=":", linewidth=2)
        ax.axvline(summ["ci_high"], color="green", linestyle=":", linewidth=2, label=f"95% CI [{summ['ci_low']:.0f}, {summ['ci_high']:.0f}]")

        p_one = np.mean(valid <= 0) if valid.size else np.nan
        ax.set_title(f"{comp} one-tailed later p={p_one:.3f}", fontweight="bold", fontsize=10)
        ax.set_xlabel("Onset difference (ms)")
        ax.set_ylabel("Frequency")
        ax.grid(alpha=0.25)
        ax.legend(fontsize=7)

    fig.suptitle(res["config_label"], fontsize=14, fontweight="bold")

    if save_path is not None:
        fig.savefig(save_path, bbox_inches="tight")
        print("Saved:", save_path)

    return fig

# Example: plot the first configuration
fig = plot_single_configuration(all_results[0])
plt.show()


## 8. One big plot: all configurations

In [ ]:

def plot_all_configurations_big(all_results, bins=40, max_rows=None, save_path=None):
    """
    Big diagnostic figure.
    Each row = one parameter configuration.
    Columns = onset CI boxplot + three bootstrap onset-difference distributions.
    """
    if max_rows is not None:
        plot_results = all_results[:max_rows]
    else:
        plot_results = all_results

    n_rows = len(plot_results)
    n_cols = 4
    fig_h = max(3.2 * n_rows, 8)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(24, fig_h), squeeze=False)

    for r, res in enumerate(plot_results):
        # --------------------------------------------------------
        # onset CI boxplot
        # --------------------------------------------------------
        ax = axes[r, 0]
        onset_dists = [res["onset_boot"][name][~np.isnan(res["onset_boot"][name])] for name in condition_names]
        box = ax.boxplot(
            onset_dists,
            labels=["GIST", "Emotion", "Scene"],
            patch_artist=True,
            widths=0.55,
            showfliers=False,
            boxprops=dict(linewidth=1),
            whiskerprops=dict(linewidth=1),
            capprops=dict(linewidth=1),
            medianprops=dict(color="black", linewidth=1.3),
        )
        for patch, name in zip(box["boxes"], condition_names):
            patch.set_facecolor(colors[name])
            patch.set_alpha(0.65)

        obs = [res["onset_obs"][name] for name in condition_names]
        ax.scatter([1, 2, 3], obs, color="darkred", s=35, marker="D", edgecolors="black", linewidths=0.8, zorder=10)
        ax.set_ylabel("Onset (ms)")
        ax.set_title("Onset CI" if r == 0 else "", fontweight="bold")
        ax.grid(axis="y", alpha=0.25)
        ax.tick_params(axis="x", labelrotation=20, labelsize=8)

        # Add compact config label on left side
        short_label = (
            f"{r+1}. base={res['baseline_mode']} | "
            f"smooth={res['smoothing_mode']}:{res['smoothing_param']} | "
            f"boot={res['bootstrap_strategy']} | "
            f"thr={res['threshold_sigma']}SD"
        )
        ax.text(
            -0.25, 0.5, short_label,
            transform=ax.transAxes,
            rotation=90,
            ha="right",
            va="center",
            fontsize=8,
            fontweight="bold",
        )

        # --------------------------------------------------------
        # difference histograms
        # --------------------------------------------------------
        for c, comp in enumerate(res["diff_boot"], start=1):
            ax = axes[r, c]
            valid = res["diff_boot"][comp][~np.isnan(res["diff_boot"][comp])]
            summ = res["diff_summary"][comp]
            obs_diff = res["diff_obs"][comp]

            ax.hist(valid, bins=bins, color="steelblue", alpha=0.75, edgecolor="black", linewidth=0.25)
            ax.axvline(0, color="black", linestyle="--", linewidth=1.1, alpha=0.7)
            ax.axvline(obs_diff, color="darkred", linewidth=1.8)
            ax.axvline(summ["ci_low"], color="green", linestyle=":", linewidth=1.4)
            ax.axvline(summ["ci_high"], color="green", linestyle=":", linewidth=1.4)
            p_one = np.mean(valid <= 0) if valid.size else np.nan

            if r == 0:
                ax.set_title(comp, fontweight="bold", fontsize=10)
            ax.text(
                0.98, 0.95,
                f"obs={obs_diff:.0f} CI=[{summ['ci_low']:.0f},{summ['ci_high']:.0f}] p={p_one:.3f}",
                transform=ax.transAxes,
                ha="right",
                va="top",
                fontsize=7,
                bbox=dict(boxstyle="round,pad=0.25", facecolor="white", edgecolor="black", alpha=0.8),
            )
            ax.grid(alpha=0.2)
            ax.set_xlabel("Diff (ms)")
            ax.set_ylabel("Freq")

    fig.suptitle(
        "Onset Sensitivity Analysis: Bootstrap CI and Onset-Difference Distributions",
        fontsize=18,
        fontweight="bold",
        y=0.995,
    )
    plt.tight_layout(rect=[0.02, 0.0, 1, 0.99])

    if save_path is not None:
        fig.savefig(save_path, bbox_inches="tight")
        print("Saved:", save_path)

    return fig

big_plot_path = os.path.join(out_dir, "all_configurations_bootstrap_CI_and_distributions.png")
fig = plot_all_configurations_big(all_results, bins=40, save_path=big_plot_path)
plt.show()


## 9. Focused diagnostic: early-onset counts

In [ ]:

early_rows = []
for res in all_results:
    base = {
        "baseline_mode": res["baseline_mode"],
        "smoothing_mode": res["smoothing_mode"],
        "smoothing_param": res["smoothing_param"],
        "bootstrap_strategy": res["bootstrap_strategy"],
        "threshold_sigma": res["threshold_sigma"],
    }
    for name in condition_names:
        boot = res["onset_boot"][name]
        valid = boot[~np.isnan(boot)]
        early_rows.append({
            **base,
            "condition": name,
            "n_valid": valid.size,
            "prop_onset_lt_50ms": np.mean(valid < 50) if valid.size else np.nan,
            "prop_onset_lt_100ms": np.mean(valid < 100) if valid.size else np.nan,
            "prop_onset_lt_200ms": np.mean(valid < 200) if valid.size else np.nan,
            "median_onset": np.nanmedian(valid) if valid.size else np.nan,
        })

early_df = pd.DataFrame(early_rows)
early_csv = os.path.join(out_dir, "early_onset_diagnostic_summary.csv")
early_df.to_csv(early_csv, index=False)
print("Saved:", early_csv)
early_df.head(20)


In [ ]:

# Show configurations with the highest proportion of very early Emotion onsets.
early_df.query("condition == 'Emotion-OTC'").sort_values("prop_onset_lt_100ms", ascending=False).head(20)



## Interpretation guide

Use the early-onset diagnostic table to identify the driver of the discrepancy.

The likely artifact pattern is:

- high `prop_onset_lt_50ms` or `prop_onset_lt_100ms` for Emotion/Scene
- wide lower CI for Emotion-GIST or Scene-GIST onset differences
- p-values changing substantially when `min_consecutive_samples`, smoothing, or baseline correction changes

If a setting produces many Emotion/Scene onsets before 100 ms, it is probably detecting transient threshold crossings rather than a stable semantic onset.
